# AutoSource Experiment Analysis

Analysis of autonomous synthetic data generation results from `results/results.tsv`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, fidelity_score, memory_gb, status, description)
df = pd.read_csv("results/results.tsv", sep="\t")
df["fidelity_score"] = pd.to_numeric(df["fidelity_score"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    score = row["fidelity_score"]
    desc = row["description"]
    print(f"  #{i:3d}  score={score:.2f}  mem={row['memory_gb']:.1f}GB  {desc}")

## Fidelity Score Over Time

Track how the best (kept) fidelity score evolves as experiments progress. The running maximum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_score = valid.loc[0, "fidelity_score"]

# Only plot points at or above baseline (the interesting region)
above = valid[valid["fidelity_score"] >= baseline_score - 1.0]

# Plot discarded as faint background dots
disc = above[above["status"] == "DISCARD"]
ax.scatter(disc.index, disc["fidelity_score"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = above[above["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["fidelity_score"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_scores = valid.loc[kept_mask, "fidelity_score"]
running_max = kept_scores.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, score in zip(kept_idx, kept_scores):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, score),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Fidelity Score (higher is better)", fontsize=12)
ax.set_title(f"AutoSource Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below baseline to above best
best_score = kept_scores.max()
margin = (best_score - baseline_score) * 0.15
ax.set_ylim(baseline_score - margin, best_score + margin)

plt.tight_layout()
plt.savefig("results/progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to results/progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_score = df.iloc[0]["fidelity_score"]
best_score = kept["fidelity_score"].max()
best_row = kept.loc[kept["fidelity_score"].idxmax()]

print(f"Baseline fidelity score:  {baseline_score:.2f}")
print(f"Best fidelity score:      {best_score:.2f}")
print(f"Total improvement:        {best_score - baseline_score:.2f} ({(best_score - baseline_score) / baseline_score * 100:.2f}%)")
print(f"Best experiment:          {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: score={row['fidelity_score']:.2f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's score
kept = df[df["status"] == "KEEP"].copy()
kept["prev_score"] = kept["fidelity_score"].shift(1)
kept["delta"] = kept["fidelity_score"] - kept["prev_score"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'Score':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.2f}  {row['fidelity_score']:.2f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.2f}  {'':>10}  TOTAL improvement over baseline")